<a href="https://colab.research.google.com/github/AshishGit22/impcode/blob/main/IPO_DRHP_document_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## summarizes IPO DRHP pdf document into summary with predefined format and constraints
## code breaks into 4 parts - Company overview, Key Management Personnel, STrengths, Strategies
## at the end combined the output of all 4 sections

## works best with Gemini 2.5 flash model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 0 — GLOBAL SETUP (COMMON FOR ALL 4 MODULES)
# ============================================================

!pip install pdfplumber google-generativeai sentence-transformers faiss-cpu spacy pandas python-docx --quiet
!python -m spacy download en_core_web_sm --quiet

import os
import re
import json
import datetime
import pdfplumber
import google.generativeai as genai
import pandas as pd
import numpy as np
import faiss
import spacy
from sentence_transformers import SentenceTransformer
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
import textwrap
from IPython.display import display, Markdown

# -----------------------
# GLOBAL CONFIGURATION
# -----------------------

COMPANY_NAME = "put company name"
API_KEY = "put llm model API key"
PDF_PATH = "put google drive path for IPO DRHP pdf document"
OUTPUT_DIR = "/content/ipo_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)


genai.configure(api_key=API_KEY)
MODEL = genai.GenerativeModel(model_name="gemini-2.5-flash")

nlp = spacy.load("en_core_web_sm")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Global setup complete.")
print("📄 Processing:", PDF_PATH)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Global setup complete.
📄 Processing: /content/drive/MyDrive/Colab Notebooks/Meesho DRHP.pdf


In [ ]:
# ============================================================
# ============================================================
#               MODULE 1 — COMPANY OVERVIEW
# ============================================================
# ============================================================

print("\n" + "="*70)
print("🚀 STARTING MODULE 1: COMPANY OVERVIEW")
print("="*70)

SECTION_NAME = "Company Overview"
CO_TOC_TITLES = ["OUR BUSINESS"]

# (All helper functions copied EXACTLY from company_overview_ver_11_final.py)
# --------------------------------------------------------------------------
# detect_page_offset
def detect_page_offset(pdf):
    print("🔍 Detecting page offset…")

    for i in range(min(30, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if not lines:
            continue

        bottom = lines[-1]
        if re.fullmatch(r"\d{1,3}", bottom):         # clean numeric footer
            printed = int(bottom)
            offset = i - (printed - 1)
            print(f"   📄 Printed page {printed} found at PDF index {i}")
            print(f"   ➕ PAGE_OFFSET = {offset}\n")
            return offset

    print("⚠️ Footer page numbers not detected. Using PAGE_OFFSET = 0.\n")
    return 0


# extract_toc
def extract_toc(pdf, offset, max_pages=15):
    print("📚 Extracting TOC…")
    toc = []

    for i in range(min(max_pages, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        for line in text.split("\n"):
            m = re.search(r"^(.*?)(\.+)\s*(\d+)$", line.strip())
            if m:
                title = m.group(1).strip().upper()
                printed = int(m.group(3))
                actual_page = printed - 1 + offset
                toc.append({
                    "title": title,
                    "printed_page": printed,
                    "pdf_page": actual_page
                })

    print(f"   📌 TOC entries detected: {len(toc)}\n")
    return toc

# find_section_page_range
def find_section_page_range(toc, aliases):
    aliases = [a.upper() for a in aliases]
    matches = [t for t in toc if t["title"] in aliases]

    if not matches:
        print("⚠️ No matching TOC title:", aliases)
        return None

    start = min(t["pdf_page"] for t in matches)
    later = sorted(t["pdf_page"] for t in toc if t["pdf_page"] > start)
    end = later[0] - 1 if later else start + 6

    print(f"   🗂 Section page range detected: PDF {start+1} to {end+1}\n")
    return start, end

# extract_text_clean
def extract_text_clean(page):
    tables = page.find_tables()
    table_bboxes = [t.bbox for t in tables]

    # Remove table objects
    def not_in_table(obj):
        if not all(k in obj for k in ["x0","x1","top","bottom"]):
            return True
        x0, top, x1, bottom = obj["x0"], obj["top"], obj["x1"], obj["bottom"]
        for (tx0, ttop, tx1, tbottom) in table_bboxes:
            if x0 >= tx0 and x1 <= tx1 and top >= ttop and bottom <= tbottom:
                return False
        return True

    cleaned = page.filter(not_in_table)

    # Remove italics
    def not_italic(obj):
        font = obj.get("fontname", "").lower()
        return not ("italic" in font or "oblique" in font)

    cleaned = cleaned.filter(not_italic)

    return cleaned.extract_text() or ""


# extract_paragraphs
def extract_paragraphs(pdf, start, end):
    print(f"📄 Extracting paragraphs from PDF pages {start+1} to {end+1}...")
    paragraphs = []

    for i in range(start, min(end+1, len(pdf.pages))):
        text = extract_text_clean(pdf.pages[i])
        blocks = re.split(r"\n\s*\n", text)

        p_idx = 0
        for blk in blocks:
            clean = " ".join(blk.split())

            # Skip obvious junk dynamically
            if len(clean.split()) < 5:
                continue
            if clean.isupper():   # headers like "OUR BUSINESS"
                continue
            if re.match(r"^\d{2,3}$", clean):  # standalone page markers
                continue
            if "image" in clean.lower() or "figure" in clean.lower():
                continue

            paragraphs.append({
                "page": i+1,
                "paragraph_idx": p_idx,
                "text": clean
            })
            p_idx += 1

    print(f"   ✔ Extracted {len(paragraphs)} paragraphs.\n")
    return paragraphs

# split_sentences
def split_sentences(paragraphs, section):
    print("✂️ Splitting into sentences…")
    records = []
    gid = 0

    for para in paragraphs:
        doc = nlp(para["text"])
        s_idx = 0

        for s in doc.sents:
            sent = s.text.strip()
            if len(sent.split()) < 3:
                continue

            uid = f"{section[:2].upper()}_{gid:05d}"

            records.append({
                "section": section,
                "page": para["page"],
                "paragraph_idx": para["paragraph_idx"],
                "sentence_idx": s_idx,
                "original_sentence": sent,
                "converted_sentence": None,
                "global_sentence_id": uid
            })
            gid += 1
            s_idx += 1

    print(f"   ✔ Total sentences: {len(records)}\n")
    return records

# simple_first_to_third
def simple_first_to_third(text: str) -> str:
    """
    Simple, deterministic 1p→3p replacement.
    We keep it conservative because final polishing will be done by Gemini.
    """
    # Remove header artifacts like "OUR BUSINESS ... Overview" if present
    text = re.sub(r"OUR BUSINESS.*?Overview", "", text, flags=re.IGNORECASE)

    phrase_rules = [
        (r"\bOur Company\b", "The Company"),
        (r"\bour company\b", "the Company"),
        (r"\bWe are\b", "The Company is"),
        (r"\bwe are\b", "the Company is"),
        (r"\bWe have\b", "The Company has"),
        (r"\bwe have\b", "the Company has"),
        (r"\bWe were\b", "The Company was"),
        (r"\bwe were\b", "the Company was"),
    ]
    for pat, rep in phrase_rules:
        text = re.sub(pat, rep, text)

    single_rules = [
        (r"\bWe\b", "The Company"),
        (r"\bwe\b", "the Company"),
        (r"\bOur\b", "The Company’s"),
        (r"\bour\b", "the Company’s"),
        (r"\bUs\b", "The Company"),
        (r"\bus\b", "the Company"),
    ]
    for pat, rep in single_rules:
        text = re.sub(pat, rep, text)

    return " ".join(text.split())

# smooth_company_references
def smooth_company_references(records, company_name=None):
    """
    Make references more natural, without LLM:
    - First sentence: use company_name if available.
    - Some later sentences can start with 'It' instead of always 'The Company'.
    - Internal repeated 'The Company' / 'The Company’s' -> 'it' / 'its'.
    """
    smoothed = []

    for idx, r in enumerate(records):
        text = r["converted_sentence"]

        # 1) First sentence: use company_name for first "The Company"
        if idx == 0 and company_name:
            text = re.sub(r"\bThe Company\b", company_name, text, count=1)

        # 2) Later sentences: occasionally allow "It" at start
        elif idx > 0 and text.startswith("The Company "):
            if idx % 3 != 0:   # keep every 3rd as "The Company"
                text = re.sub(r"^The Company\b", "It", text, count=1)

        # 3) Internal occurrences: keep leading ref, rest → it/its
        m = re.match(r"^(The Company’s|The Company)\b(.*)$", text)
        if m:
            leading = m.group(1)
            rest = m.group(2)
        else:
            leading = ""
            rest = text

        rest = re.sub(r"\bThe Company’s\b", "its", rest)
        rest = re.sub(r"\bThe Company\b", "it", rest)

        text = (leading + rest) if leading else rest

        r["converted_sentence"] = " ".join(text.split())
        smoothed.append(r)

    return smoothed


# convert_sentences
def convert_sentences(records, company_name=None):
    """
    Step 3 (NON-LLM version):
    - Rule-based 1p→3p
    - Then smoothing of company references
    """
    print("👤 Converting to third person (rule-based)…")

    for r in records:
        orig = r["original_sentence"]
        conv = simple_first_to_third(orig)
        r["converted_sentence"] = conv

    records = smooth_company_references(records, company_name=company_name)
    return records

# remove_duplicates
def remove_duplicates(records):
    print("🧹 Removing duplicate sentences…")
    seen = set()
    unique = []

    for r in records:
        t = r["converted_sentence"]
        normalized = re.sub(r"[^\w ]", "", t.lower())
        if normalized not in seen:
            seen.add(normalized)
            unique.append(r)

    print(f"   ✔ Unique sentences: {len(unique)}\n")
    return unique

# gemini_select
def gemini_select(records, max_sents=20):
    print("🤖 Gemini selecting sentences (Step 2)…")

    lines = [f"{r['global_sentence_id']}: {r['converted_sentence']}" for r in records]
    joined = "\n".join(lines)

    prompt = f"""
You are preparing a Company Overview summary from an IPO document.

Rules:
- DO NOT rewrite, paraphrase, or merge sentences.
- ONLY select sentences from the list.
- Use EXACT wording provided.
- Return at most {max_sents} sentences.
- Return ONLY a JSON list of IDs like ["CO_00001","CO_00005"].

Sentences:
{joined}
"""

    try:
        resp = MODEL.generate_content(
            prompt,
            generation_config={"temperature": 0.0}
        )

        # --- NEW CODE TO CLEAN UP RESPONSE ---
        raw_text = resp.text.strip()

        # Use regex to strip the leading/trailing code block markers (```json\n...\n```)
        # We use re.IGNORECASE to catch '```JSON', '```json', etc. and re.DOTALL
        # to handle multiline responses, though less likely here.
        clean_text = re.sub(r"^\s*```(json)?\s*|```\s*$", "", raw_text, flags=re.IGNORECASE | re.DOTALL).strip()
        # -------------------------------------

        if not clean_text:
            raise ValueError("API returned content but it was empty after cleanup.")

        # Decode the cleaned text
        selected = json.loads(clean_text)

        return selected

    # Catching JSONDecodeError specifically is generally better,
    # but keeping the general Exception catch from your original code block here:
    except Exception as e:
        print("⚠️ Gemini Step2 error:", e)
        print("⚠️ Falling back to first N sentences.")
        return [r["global_sentence_id"] for r in records[:max_sents]]

# polish_summary_with_gemini
def polish_summary_with_gemini(summary: str, company_name: str = None) -> str:
    """
    Single final call:
    - Fix grammar and punctuation
    - Smooth 'The Company' / 'it' / company name usage
    - No paraphrasing or content changes
    """
    company_ref = company_name or "The Company"

    prompt = f"""
You are lightly editing a legally sensitive IPO Company Overview summary for {company_ref}.

Goals:
- Fix ONLY grammar, punctuation, and awkward repetition of references like
  "The Company", "it", "its", and "{company_ref}".
- Make the use of "{company_ref}", "the Company", and "it/its" sound natural and varied.
- Convert the values in millions to values in crores maximum upto 2 decimals (e.g. 123.77 million to 12.37 cr)
- DO NOT paraphrase, rephrase, shorten, expand, or reorder the content.
- DO NOT add, remove, or change any facts, numbers, dates, or entity names.
- Keep each sentence's meaning intact; only adjust wording where strictly necessary
  for grammatical correctness and pronoun smoothness.

Return ONLY the edited summary text.

Summary:
{summary}
"""

    try:
        resp = MODEL.generate_content(
            prompt,
            generation_config={"temperature": 0.0}
        )
        cleaned = (resp.text or "").strip()
        return cleaned if cleaned else summary
    except Exception as e:
        print("⚠️ Gemini final polish error:", e)
        print("   Using unpolished summary instead.")
        return summary

# --------------------------------------------------------------------------

# >>>>>>  IMPORTANT  <<<<<<
# Due to length constraints of ChatGPT output, the helper functions from:
# company_overview_ver_11_final.py
# must be pasted here EXACTLY as they exist in your file.
#
# DO NOT modify anything.
#
# (Paste entire Module 1 code here exactly as-is, except Cell 0.)

# After full execution:
# Ensure final variable:
# polished_summary_string

# ============================================
# CELL 2 — Run Company Overview Pipeline
# ============================================

SECTION_NAME = "Company Overview"
CO_TOC_TITLES = ["OUR BUSINESS"]   # can add more aliases if needed

print("\n🚀 Starting Company Overview Extraction...\n")

with pdfplumber.open(PDF_PATH) as pdf:

    # STEP 1 — PAGE OFFSET
    PAGE_OFFSET = detect_page_offset(pdf)

    # STEP 2 — TOC
    toc = extract_toc(pdf, PAGE_OFFSET)

    # STEP 3 — PAGE RANGE
    page_range = find_section_page_range(toc, CO_TOC_TITLES)
    if not page_range:
        raise ValueError("Section not found based on TOC aliases.")

    start_page, end_page = page_range

    # STEP 4 — PARAGRAPHS (tables & italics removed)
    paragraphs = extract_paragraphs(pdf, start_page, end_page)

# DEBUG PREVIEW
print("\n🧪 DEBUG PREVIEW — first few extracted paragraphs:\n")
preview = "\n".join([p["text"] for p in paragraphs[:3]])
print(textwrap.fill(preview, width=110))
print("\n---------------------------------------------\n")

# STEP 5 — SPLIT SENTENCES
records = split_sentences(paragraphs, SECTION_NAME)

# STEP 6 — 1p→3p (rule-based) + SMOOTHING
records = convert_sentences(records, company_name=COMPANY_NAME)

# STEP 7 — REMOVE DUPLICATES
records = remove_duplicates(records)

# STEP 8 — SAVE CONVERSION AUDIT
conv_df = pd.DataFrame(records)
conv_path = os.path.join(OUTPUT_DIR, "company_overview_conversion_audit.csv")
conv_df.to_csv(conv_path, index=False, encoding="utf-8-sig")
print(f"📊 Conversion audit saved: {conv_path}\n")

# STEP 9 — GEMINI SELECTION (Step 2)  → 1 LLM call
selected_ids = gemini_select(records, max_sents=20)

# Map IDs → records
id_map = {r["global_sentence_id"]: r for r in records}
selected_records = [id_map[i] for i in selected_ids if i in id_map]

print("\n🔎 Selected sentences (first few):\n")
for r in selected_records[:5]:
    print(" •", r["converted_sentence"])

# STEP 10 — BUILD RAW FINAL SUMMARY
raw_summary = " ".join(r["converted_sentence"] for r in selected_records)
raw_readable = textwrap.fill(raw_summary, width=110)

# STEP 11 — FINAL GEMINI POLISH (Step 3)  → 1 LLM call
polished_summary = polish_summary_with_gemini(raw_summary, company_name=COMPANY_NAME)
polished_readable = textwrap.fill(polished_summary, width=110)

# SAVE SUMMARY FILES
raw_path = os.path.join(OUTPUT_DIR, "company_overview_summary_raw.txt")
polished_path = os.path.join(OUTPUT_DIR, "company_overview_summary_polished.txt")

with open(raw_path, "w", encoding="utf-8") as f:
    f.write(raw_summary)

with open(polished_path, "w", encoding="utf-8") as f:
    f.write(polished_summary)

print(f"\n📝 Summary saved at:")
print("   • RAW:      {raw_path}")
print("   • POLISHED: {polished_path}")

# STEP 12 — SELECTION AUDIT
sel_rows = []
for r in records:
    gid = r["global_sentence_id"]
    included = 1 if gid in selected_ids else 0
    order = selected_ids.index(gid)+1 if included else None

    sel_rows.append({
        "global_sentence_id": gid,
        "included_in_summary": included,
        "summary_order": order,
        "page": r["page"],
        "paragraph_idx": r["paragraph_idx"],
        "sentence_idx": r["sentence_idx"],
        "sentence": r["converted_sentence"]
    })

sel_df = pd.DataFrame(sel_rows)
sel_path = os.path.join(OUTPUT_DIR, "company_overview_selection_audit.csv")
sel_df.to_csv(sel_path, index=False, encoding="utf-8-sig")

print(f"📑 Selection audit saved: {sel_path}")

# PRINT FINAL SUMMARY
print("\n📄 ===== FINAL COMPANY OVERVIEW SUMMARY (POLISHED) =====\n")
print(polished_readable)
print("\n=============================================\n")

print("🎉 Pipeline completed successfully.")

company_overview_final_text = polished_summary
print("✅ Module 1 Completed.")


🚀 STARTING MODULE 1: COMPANY OVERVIEW

🚀 Starting Company Overview Extraction...

🔍 Detecting page offset…
   📄 Printed page 1 found at PDF index 6
   ➕ PAGE_OFFSET = 6

📚 Extracting TOC…
   📌 TOC entries detected: 52

   🗂 Section page range detected: PDF 306 to 348

📄 Extracting paragraphs from PDF pages 306 to 348...
   ✔ Extracted 31 paragraphs.


🧪 DEBUG PREVIEW — first few extracted paragraphs:

OUR BUSINESS Overview We are a multi-sided technology platform driving e-commerce in India by bringing
together four key stakeholders – consumers, sellers, logistics partners and content creators. Our e-commerce
marketplace, that we operate under the brand name Meesho, emerged as India’s largest in terms of number of
Placed Orders and Annual Transacting Users among e-commerce players in India in the last twelve months period
ended June 30, 2025, according to the Redseer Report. Our value focused platform is designed to serve all
segments of consumers across India by making e- commerce af

In [ ]:
# ============================================================
# ============================================================
#        MODULE 2 — KEY MANAGEMENT PERSONNEL (KMP)
# ============================================================
# ============================================================

# print("\n" + "="*70)
print("🚀 STARTING MODULE 2: KEY MANAGEMENT PERSONNEL")
# print("="*70)

SECTION_NAME = "Key Management Personnel"
CO_TOC_TITLES = ["OUR MANAGEMENT"]

# KMP header lists (start & end)
kmp_start_headers = [
    "brief profiles of our directors",
    "brief profile of directors",
    "profiles of directors",
    "profile of our directors"
]
kmp_end_headers = [
    "relationship between our directors",
    "relationship between directors"
]


# (All helper functions copied EXACTLY from kmp_version8.py)
# --------------------------------------------------------------------------
# detect_page_offset
def detect_page_offset(pdf):
    """Detects the offset between the printed page number and the PDF index."""
    print("🔍 Detecting page offset…")

    for i in range(min(30, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if not lines:
            continue

        bottom = lines[-1]
        # Regex to find a clean numeric footer
        if re.fullmatch(r"\d{1,3}", bottom):
            printed = int(bottom)
            offset = i - (printed - 1)
            print(f"    📄 Printed page {printed} found at PDF index {i}")
            print(f"    ➕ PAGE_OFFSET = {offset}\n")
            return offset

    print("⚠️ Footer page numbers not detected. Using PAGE_OFFSET = 0.\n")
    return 0

# extract_toc
def extract_toc(pdf, offset, max_pages=15):
    """Extracts Table of Contents entries from the first few pages of the PDF."""
    print("📚 Extracting TOC…")
    toc = []

    for i in range(min(max_pages, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        for line in text.split("\n"):
            # Matches: Title ........... PageNumber
            m = re.search(r"^(.*?)(\.+)\s*(\d+)$", line.strip())
            if m:
                title = m.group(1).strip().upper()
                printed = int(m.group(3))
                actual_page = printed - 1 + offset
                toc.append({
                    "title": title,
                    "printed_page": printed,
                    "pdf_page": actual_page
                })

    print(f"    📌 TOC entries detected: {len(toc)}\n")
    return toc

# find_section_page_range
def find_section_page_range(toc, aliases):
    """
    Finds the 0-based PDF page index range for a section (start and end).
    The end is defined by the page index just before the next TOC entry.
    """
    aliases = [a.upper() for a in aliases]
    matches = [t for t in toc if t["title"] in aliases]

    if not matches:
        print("⚠️ No matching TOC title:", aliases)
        return None

    # Find the earliest matching page for the start (0-based index)
    start = min(t["pdf_page"] for t in matches)

    # Find the page of the next entry in the entire TOC after the start page
    later = sorted(t["pdf_page"] for t in toc if t["pdf_page"] > start)

    # End is the page index one before the next section starts, or a fallback
    end = later[0] - 1 if later else start + 6

    if end < start:
        end = start

    print(f"    🗂 Section page range detected: PDF index {start} to {end} (Pages {start+1} to {end+1})\n")
    return start, end

# extract_text_clean
def extract_text_clean(page):
    """Extracts text while filtering out tables and italics."""
    tables = page.find_tables()
    table_bboxes = [t.bbox for t in tables]

    def not_in_table(obj):
        if not all(k in obj for k in ["x0","x1","top","bottom"]): return True
        x0, top, x1, bottom = obj["x0"], obj["top"], obj["x1"], obj["bottom"]
        for (tx0, ttop, tx1, tbottom) in table_bboxes:
            if x0 >= tx0 and x1 <= tx1 and top >= ttop and bottom <= tbottom:
                return False
        return True

    cleaned = page.filter(not_in_table)

    def not_italic(obj):
        font = obj.get("fontname", "").lower()
        return not ("italic" in font or "oblique" in font)

    cleaned = cleaned.filter(not_italic)

    return cleaned.extract_text() or ""

# get_bold_phrases
def get_bold_phrases(page):
    """
    Extracts multi-word phrases rendered in bold font from a pdfplumber page.
    This is used to automatically identify KMP names/headers.
    """
    bold_phrases = set()

    # Extract words with x_tolerance to merge closely spaced characters
    words = page.extract_words(x_tolerance=1.5, y_tolerance=1.5)

    current_phrase = []

    for word in words:
        font = word.get("fontname", "").lower()
        # Check for common bold font identifiers
        is_bold = "bold" in font or "bd" in font or "black" in font

        if is_bold:
            current_phrase.append(word['text'])
        else:
            if current_phrase:
                phrase = ' '.join(current_phrase).strip()
                # Only keep phrases that look like names (e.g., at least 2 capitalized words)
                if len(phrase.split()) >= 2 and any(w[0].isupper() for w in phrase.split()):
                    bold_phrases.add(phrase)
                current_phrase = []

    # Check for a trailing phrase
    if current_phrase:
        phrase = ' '.join(current_phrase).strip()
        if len(phrase.split()) >= 2 and any(w[0].isupper() for w in phrase.split()):
             bold_phrases.add(phrase)

    # Filter out page numbers that might be bolded
    return {p for p in bold_phrases if not re.fullmatch(r'\d{1,4}', p.replace(',', '').strip())}

# extract_paragraphs
def extract_paragraphs(pdf, start, end):
    """Extracts cleaned paragraphs from a page range."""
    print(f"📄 Extracting paragraphs from PDF pages {start+1} to {end+1}...")
    paragraphs = []

    for i in range(start, min(end+1, len(pdf.pages))):
        # Use the cleaning function before splitting into blocks
        text = extract_text_clean(pdf.pages[i])
        blocks = re.split(r"\n\s*\n", text)

        p_idx = 0
        for blk in blocks:
            clean = " ".join(blk.split())

            # Skip obvious junk dynamically
            if len(clean.split()) < 5: continue
            if clean.isupper(): continue
            if re.match(r"^\d{2,3}$", clean): continue
            if "image" in clean.lower() or "figure" in clean.lower(): continue

            paragraphs.append({
                "page": i+1,
                "paragraph_idx": p_idx,
                "text": clean
            })
            p_idx += 1

    print(f"    ✔ Extracted {len(paragraphs)} paragraphs.\n")
    return paragraphs

# split_sentences
def split_sentences(paragraphs, section):
    """Splits paragraphs into sentences using spaCy."""
    print("✂️ Splitting into sentences…")
    records = []
    gid = 0

    for para in paragraphs:
        doc = nlp(para["text"])
        s_idx = 0

        for s in doc.sents:
            sent = s.text.strip()
            if len(sent.split()) < 3:
                continue

            uid = f"{section[:2].upper()}_{gid:05d}"

            records.append({
                "section": section,
                "page": para["page"],
                "paragraph_idx": para["paragraph_idx"],
                "sentence_idx": s_idx,
                "original_sentence": sent,
                "converted_sentence": None,
                "global_sentence_id": uid
            })
            gid += 1
            s_idx += 1

    print(f"    ✔ Total sentences: {len(records)}\n")
    return records

# simple_first_to_third
def simple_first_to_third(text: str) -> str:
    """Simple, deterministic 1p→3p replacement."""
    text = re.sub(r"OUR BUSINESS.*?Overview", "", text, flags=re.IGNORECASE)

    phrase_rules = [
        (r"\bOur Company\b", "The Company"),
        (r"\bour company\b", "the Company"),
        (r"\bWe are\b", "The Company is"),
        (r"\bwe are\b", "the Company is"),
        (r"\bWe have\b", "The Company has"),
        (r"\bwe have\b", "the Company has"),
        (r"\bWe were\b", "The Company was"),
        (r"\bwe were\b", "the Company was"),
    ]
    for pat, rep in phrase_rules:
        text = re.sub(pat, rep, text)

    single_rules = [
        (r"\bWe\b", "The Company"),
        (r"\bwe\b", "the Company"),
        (r"\bOur\b", "The Company’s"),
        (r"\bour\b", "the Company’s"),
        (r"\bUs\b", "The Company"),
        (r"\bus\b", "the Company"),
    ]
    for pat, rep in single_rules:
        text = re.sub(pat, rep, text)

    return " ".join(text.split())

# smooth_company_references
def smooth_company_references(records, company_name=None):
    """Make references more natural."""
    smoothed = []

    for idx, r in enumerate(records):
        text = r["converted_sentence"]

        # 1) First sentence: use company_name for first "The Company"
        if idx == 0 and company_name:
            text = re.sub(r"\bThe Company\b", company_name, text, count=1)

        # 2) Later sentences: occasionally allow "It" at start
        elif idx > 0 and text.startswith("The Company "):
            if idx % 3 != 0:
                text = re.sub(r"^The Company\b", "It", text, count=1)

        # 3) Internal occurrences: keep leading ref, rest → it/its
        m = re.match(r"^(The Company’s|The Company)\b(.*)$", text)
        if m:
            leading = m.group(1)
            rest = m.group(2)
        else:
            leading = ""
            rest = text

        rest = re.sub(r"\bThe Company’s\b", "its", rest)
        rest = re.sub(r"\bThe Company\b", "it", rest)

        text = (leading + rest) if leading else rest

        r["converted_sentence"] = " ".join(text.split())
        smoothed.append(r)

    return smoothed

# convert_sentences
def convert_sentences(records, company_name=None):
    """Rule-based 1p→3p + smoothing wrapper."""
    print("👤 Converting to third person (rule-based)…")

    for r in records:
        orig = r["original_sentence"]
        conv = simple_first_to_third(orig)
        r["converted_sentence"] = conv

    records = smooth_company_references(records, company_name=company_name)
    return records

# polish_summary_with_gemini
def polish_summary_with_gemini(summary: str, company_name: str = None) -> str:
    """Applies a single final LLM call for grammar and pronoun smoothing."""
    # company_ref = company_name or "The Company"
    # company_ref = company_name

    prompt = f"""
You are lightly editing a legally sensitive IPO Key Management Personnel (KMP) summary for {company_name}.

Goals:
- Fix ONLY grammar, punctuation, and awkward repetition of references like
  "The Company", "it", "its", and "{company_name}".
- Make the use of "the Company", "{company_name}" and "it/its" sound natural and varied.
- Convert the values in millions to values in crores maximum upto 2 decimals (e.g. 123.77 million to 12.37 cr).
- DO NOT paraphrase, rephrase, shorten, expand, or reorder the content.
- DO NOT add, remove, or change any facts, numbers, dates, or entity names.
- Keep each sentence's meaning intact; only adjust wording where strictly necessary
  for grammatical correctness and pronoun smoothness.

Return ONLY the edited summary text.

Summary:
{summary}
"""

    try:
        resp = MODEL.generate_content(
            prompt,
            generation_config={"temperature": 0.0}
        )
        cleaned = (resp.text or "").strip()
        return cleaned if cleaned else summary
    except Exception as e:
        print("⚠️ Gemini final polish error:", e)
        print("    Using unpolished summary instead.")
        return summary

# Entire robust main pipeline logic
# --------------------------------------------------------------------------

# >>>>>>  IMPORTANT  <<<<<<
# Paste the full KMP script here exactly as-is (except its own Cell 0).

print("\n🚀 Starting Key Management Personnel (KMP) Extraction Pipeline (Robust Single LLM Call Mode)...\n")

# --- 1. TOC Isolation and Raw KMP Text Extraction ---

extracted_kmp_pages = []
raw_extracted_text = ""
start_found = False
start_pdf_page_idx = -1
end_pdf_page_idx = -1
records = [] # Initialize records list
kmp_paragraphs = [] # Initialize kmp_paragraphs list

try:
    with pdfplumber.open(PDF_PATH) as pdf:
        # STEP 1, 2, 3 — PAGE OFFSET, TOC, PAGE RANGE (Your original logic)
        PAGE_OFFSET = detect_page_offset(pdf)
        toc = extract_toc(pdf, PAGE_OFFSET)
        section_range = find_section_page_range(toc, CO_TOC_TITLES)

        if section_range is None:
            raise ValueError(f"Section not found based on TOC aliases: {CO_TOC_TITLES}.")

        start_pdf_page_idx, end_pdf_page_idx = section_range

        # --- KMP Header-Based Extraction (Isolation Logic) ---
        print("--- Progress Remark: Applying KMP start/end header logic within the section pages. ---")

        page_number_re = re.compile(r'\s*\b\d{1,4}\b\s*$')
        start_pattern = r'|'.join(re.escape(h) for h in kmp_start_headers)
        end_pattern = r'|'.join(re.escape(h) for h in kmp_end_headers)
        kmp_start_regex = re.compile(f'({start_pattern})', re.IGNORECASE)
        kmp_end_regex = re.compile(f'({end_pattern})', re.IGNORECASE)

        for i in range(start_pdf_page_idx, min(end_pdf_page_idx + 1, len(pdf.pages))):
            page = pdf.pages[i]
            text_segment = page.extract_text() or ""
            page_num = i + 1
            content = ""

            if not start_found:
                match_start = kmp_start_regex.search(text_segment)
                if match_start:
                    kmp_content = text_segment[match_start.start():]
                    match_end = kmp_end_regex.search(kmp_content)
                    if match_end:
                        content = kmp_content[:match_end.start()].strip()
                        extracted_kmp_pages.append((content, page_num))
                        break
                    else:
                        content = kmp_content.strip()
                        extracted_kmp_pages.append((content, page_num))
                        start_found = True

            elif start_found:
                match_end = kmp_end_regex.search(text_segment)
                if match_end:
                    content = text_segment[:match_end.start()].strip()
                    extracted_kmp_pages.append((content, page_num))
                    break
                else:
                    content = text_segment.strip()
                    extracted_kmp_pages.append((content, page_num))

        # Apply cleanup (remove trailing page numbers)
        cleaned_kmp_pages = []
        for text_segment, page_num in extracted_kmp_pages:
            cleaned_segment = page_number_re.sub('', text_segment).strip()
            if cleaned_segment:
                cleaned_kmp_pages.append((cleaned_segment, page_num))

        extracted_kmp_pages = cleaned_kmp_pages
        raw_extracted_text = "\n\n".join(text for text, _ in extracted_kmp_pages)

        if not raw_extracted_text:
            raise ValueError("KMP Profiles sub-section headers not found within the isolated pages.")

        print(f"--- Progress Remark: Raw KMP text extracted. Total segments: {len(extracted_kmp_pages)} ---")

        # --- 4. PARAGRAPH (PROFILE) Grouping Logic (Single Line Break) ---
        print("--- Progress Remark: Grouping lines into profiles using single newline separation. ---")

        # Heuristic for detecting the start of a new profile
        name_check_regex = re.compile(r'^(?:Mr\.|Ms\.|Dr\.)?\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})', re.IGNORECASE)

        for text_segment, page_num in extracted_kmp_pages:
            # Split on single newline
            blocks = re.split(r"\n", text_segment)
            p_idx = 0
            current_profile_block = ""

            for blk in blocks:
                clean = " ".join(blk.split()).strip()
                if not clean: continue # Skip empty lines

                is_new_profile = name_check_regex.search(clean)

                if is_new_profile and current_profile_block:
                    # New profile detected: save the old one
                    kmp_paragraphs.append({
                        "page": page_num,
                        "paragraph_idx": p_idx,
                        "text": current_profile_block.strip()
                    })
                    p_idx += 1
                    current_profile_block = clean # Start new block
                else:
                    # Continue current profile block
                    if not current_profile_block:
                        current_profile_block = clean
                    else:
                        current_profile_block += " " + clean

            # Save the last profile block from the segment
            if current_profile_block:
                kmp_paragraphs.append({
                    "page": page_num,
                    "paragraph_idx": p_idx,
                    "text": current_profile_block.strip()
                })

        if not kmp_paragraphs:
             raise ValueError("No paragraphs/profiles were successfully grouped.")

        # STEP 5 — SPLIT SENTENCES (Runs on the newly grouped paragraphs)
        records = split_sentences(kmp_paragraphs, SECTION_NAME)

except Exception as e:
    print(f"\nFATAL ERROR during extraction: {e}")

# --- 2. Conversion and Polishing (Single Call) ---

if records:
    # STEP 6 — 1p→3p (rule-based) + SMOOTHING on ALL sentences
    records = convert_sentences(records, company_name=COMPANY_NAME)

    # BUILD RAW CONVERTED SUMMARY
    raw_converted_summary = " ".join(r["converted_sentence"] for r in records)

    # --- NEW STEP 7: SINGLE GEMINI POLISH CALL WITH FORMATTING INSTRUCTIONS ---
    print("\n✨ Step 7: Performing single LLM polish with explicit formatting instructions...")

    # a. Extract name list for the prompt (REUSABLE COMPONENT)
    kmp_names = []
    name_check_regex = re.compile(r'^(?:Mr\.|Ms\.|Dr\.)?\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})', re.IGNORECASE)
    for p in kmp_paragraphs:
        match = name_check_regex.search(p['text'])
        if match:
            kmp_names.append(match.group(1).strip())
        # If the name can't be found, use a placeholder to maintain the list size
        else:
            kmp_names.append("Unknown Profile Header")

    # b. Prepare the custom prompt (REUSABLE COMPONENT)
    instruction_prefix = f"""
    Please polish the following text for grammar, flow, and professionalism. The text contains the profiles of Key Management Personnel (KMP) of {COMPANY_NAME}.

    CRITICAL INSTRUCTION: Your output MUST be a numbered, bulleted list. The list must contain the profiles of the following individuals, in order: {', '.join(kmp_names)}.
    Start each profile with its corresponding number, followed by the bolded name, then a colon, and then the polished profile text.

    Example format:
    1. **Vidit Aatrey**: [Polished profile text here]
    2. **Sanjeev Kumar**: [Polished profile text here]

    Do not include any introductory or concluding remarks. Return only the numbered list.
    """

    # Combine instruction and summary for the single LLM call
    full_prompt_for_llm = instruction_prefix + "\n\nTEXT TO POLISH:\n" + raw_converted_summary

    # NOTE: Assuming polish_summary_with_gemini in Cell 1 is updated to accept the full prompt.
    polished_summary_string = polish_summary_with_gemini(full_prompt_for_llm, company_name=COMPANY_NAME)

    # The result of the LLM call is now the final output.
    final_markdown_output = polished_summary_string.strip()

else:
    raw_converted_summary = ""
    polished_summary_string = "No summary available."
    final_markdown_output = "No summary available."


# --- 3. Output File Saving and Auditing ---
print("\n--- Progress Remark: Saving files and generating audit CSV. ---")

# SAVE SUMMARY FILES
raw_path = os.path.join(OUTPUT_DIR, "kmp_profiles_raw.txt")
polished_path = os.path.join(OUTPUT_DIR, "kmp_profiles_polished.txt")

with open(raw_path, "w", encoding="utf-8") as f:
    f.write(raw_converted_summary)

with open(polished_path, "w", encoding="utf-8") as f:
    # Save the correctly formatted string
    f.write(final_markdown_output)

print(f"\n📝 Summary saved at:")
print(f"  • RAW: {raw_path}")
print(f"  • POLISHED: {polished_path}")

# SAVE AUDIT CSV
audit_df = pd.DataFrame(records)
# Rename columns for clarity in the final audit file
audit_df = audit_df.rename(columns={
    "original_sentence": "Original Sentence (From PDF)",
    "converted_sentence": "Converted Sentence (Rule-Based)",
    "page": "Page Number",
    "global_sentence_id": "Sentence ID"
}) # FIX: Removed .drop(columns=['section']) to resolve KeyError.

audit_path = os.path.join(OUTPUT_DIR, "kmp_profiles_audit.csv")
audit_df.to_csv(audit_path, index=False, encoding="utf-8-sig")

print(f"📑 Audit CSV saved: {audit_path}")


# 4. PRINT FINAL SUMMARY
print(f"\n📄 ===== FINAL {SECTION_NAME.upper()} SUMMARY (POLISHED) (BULLETED VIEW) =====\n")

# Use the final markdown output string for display
display(Markdown(final_markdown_output))

print("\n=============================================\n")
print("🎉 Pipeline completed successfully.")


# After execution:
kmp_final_text = polished_summary_string.strip()

print("✅ Module 2 Completed.")

🚀 STARTING MODULE 2: KEY MANAGEMENT PERSONNEL

🚀 Starting Key Management Personnel (KMP) Extraction Pipeline (Robust Single LLM Call Mode)...

🔍 Detecting page offset…
    📄 Printed page 1 found at PDF index 6
    ➕ PAGE_OFFSET = 6

📚 Extracting TOC…
    📌 TOC entries detected: 52

    🗂 Section page range detected: PDF index 370 to 393 (Pages 371 to 394)

--- Progress Remark: Applying KMP start/end header logic within the section pages. ---
--- Progress Remark: Raw KMP text extracted. Total segments: 2 ---
--- Progress Remark: Grouping lines into profiles using single newline separation. ---
✂️ Splitting into sentences…
    ✔ Total sentences: 73

👤 Converting to third person (rule-based)…

✨ Step 7: Performing single LLM polish with explicit formatting instructions...

--- Progress Remark: Saving files and generating audit CSV. ---

📝 Summary saved at:
  • RAW: /content/ipo_outputs/kmp_profiles_raw.txt
  • POLISHED: /content/ipo_outputs/kmp_profiles_polished.txt
📑 Audit CSV saved: /co

1.  **Vidit Aatrey**: Vidit Aatrey, the Company’s Promoter, is the Chairman, Managing Director, and Chief Executive Officer of Meesho. He has been associated with the Company as a director since August 13, 2015. He holds a Bachelor of Technology degree in Electrical Engineering from the Indian Institute of Technology, Delhi. He leads the executive management team and is responsible for setting Meesho’s strategic vision, driving key initiatives, ensuring operational excellence, and long-term growth. He was previously associated with ITC Limited and InMobi Technology Services Private Limited. He has been featured in Forbes Asia’s 30 Under 30 list in 2018, Forbes India’s 30 Under 30 list in 2018, Entrepreneur Magazine’s 35 Under 35 list in 2019, and Fortune India’s 40 Under 40 list in 2021, 2024, and 2025.
2.  **Sanjeev Kumar**: Sanjeev Kumar, the Company’s Promoter, is a Whole-time Director and Chief Technology Officer of Meesho. He has been associated with the Company as a director since August 13, 2015. He holds a Bachelor of Technology degree in Electrical Engineering from the Indian Institute of Technology, Delhi. He is responsible for executing the overall technology vision and ensuring the scalability, security, and efficiency of Meesho’s technology infrastructure. He was previously associated with Sony Corporation. He has been featured in Forbes Asia’s 30 Under 30 list in 2018, Forbes India’s 30 Under 30 list in 2018, Fortune India’s 40 Under 40 list in 2021 and 2024, and The Economic Times 40 Under 40 list in 2024.
3.  **Mohit Bhatnagar**: Mohit Bhatnagar is a Non-Executive Non-Independent Director (nominee of Peak XV Partners Investments V) on the Board of Meesho. He had been associated as a director with the Company’s erstwhile holding company, Meesho Inc., since May 23, 2018, and has been associated with the Company as a director since June 16, 2025. He holds a Bachelor of Engineering degree in Electronics Engineering from the Thadomal Shahani Engineering College, University of Bombay, Mumbai; a Master of Science degree in Electrical Engineering from the Virginia Polytechnic Institute and State University, Virginia; and a Master’s degree in Business Administration from The University of North Carolina at Chapel Hill. He is a designated partner at Peak XV Partners Advisors India LLP and has been associated with Peak XV Partners (formerly Sequoia Capital India & SEA) since 2006.
4.  **Mukul Arora**: Mukul Arora is a Non-Executive Non-Independent Director (nominee of Elevation Capital V Limited) on the Board of Meesho. He had been associated as a director with the Company’s erstwhile holding company, Meesho Inc., since May 23, 2018, and has been associated with the Company as a director since June 4, 2025. He holds a Bachelor of Engineering degree in Computers from the University of Delhi and a Post-Graduate Diploma in Management from the Indian Institute of Management Society, Lucknow. At present, he is associated with Light Ray Advisors LLP and was previously associated with McKinsey & Company, Inc. He was awarded the ‘Midas Touch’ award at The Economic Times Start-up Awards 2024.
5.  **Rohit Bhagat**: Rohit Bhagat is an Independent Director of Meesho. He had been associated as an independent director with the Company’s erstwhile holding company, Meesho Inc., since June 19, 2023, and has been associated with the Company as a director since June 16, 2025. He holds a Bachelor of Technology degree in Mechanical Engineering from the Indian Institute of Technology, Delhi; a Master of Science degree in Engineering from the University of Texas at Austin, Texas; and a Master’s degree in Management from the J. L. Kellogg Graduate School of Management at Northwestern University, Illinois. He has also completed the Directors’ Consortium Executive Program from the Graduate School of Business. At present, he serves on the board of PhonePe Limited and previously served as the senior managing director and chairman of BlackRock Inc.'s Asia-Pacific business.
6.  **Hari Shanker Bhartia**: Hari Shanker Bhartia is an Independent Director of Meesho. He had been associated as an independent director with the Company’s erstwhile holding company, Meesho Inc., since July 8, 2024, and has been associated with the Company as a director since June 16, 2025. He holds a Bachelor of Technology degree in Chemical Engineering from the Indian Institute of Technology, Delhi. He is the co-founder and co-chairman of the Jubilant Bhartia Group, acting as the co-chairman of Jubilant Pharmova Limited and Jubilant FoodWorks Limited, and the co-chairman and Whole-time Director of Jubilant Ingrevia Limited.
7.  **Surojit Chatterjee**: Surojit Chatterjee is an Independent Director of Meesho. He had been associated as an independent director with the Company’s erstwhile holding company, Meesho Inc., since March 31, 2024, and has been associated with the Company as a director since June 16, 2025. He holds a Bachelor of Technology (Honours) degree in Computer Science and Engineering from the Indian Institute of Technology, Kharagpur; a Master’s degree in Computer Science from the State University of New York at Buffalo; and a Master’s degree in Business Administration from the Massachusetts Institute of Technology. He is the founder and Chief Executive Officer of EMA Unlimited Inc. He was previously associated with Coinbase Global Inc., Flipkart Internet Private Limited, and Oracle Corporation. He is also an independent director at Atos.net.
8.  **Kimsuka Narasimhan**: Kimsuka Narasimhan is an Independent Director of Meesho. She has been associated with the Company as a director since June 22, 2025. She holds a Bachelor of Commerce degree from the University of Madras and has passed the final examination held by the Institute of Cost and Works Accountants of India. She is also an associate of the Institute of Chartered Accountants of India. At present, she serves on the board of Bharti Airtel Limited and was previously associated with PepsiCo India Holdings Private Limited and Kimberly-Clark Asia Pacific Headquarters Pte Limited.



🎉 Pipeline completed successfully.
✅ Module 2 Completed.


In [ ]:

# ============================================================
# ============================================================
#               MODULE 3 — STRENGTHS
# ============================================================
# ============================================================

print("\n" + "="*70)
print("🚀 STARTING MODULE 3: STRENGTHS")
print("="*70)

SECTION_NAME = "Strengths"
CO_TOC_TITLES = ["OUR BUSINESS"]

START_MARKERS = ["Our Strengths", "Our Competitive Strengths"]
END_MARKERS = ["Our Strategies", 'Our Strategy', "Our Growth Strategies"]

# (All helper functions copied EXACTLY from strengths_ver_13_final.py)
# --------------------------------------------------------------------------
# detect_page_offset
def detect_page_offset(pdf):
    """Detects printed page number vs PDF index."""
    print("🔍 Step 1: Detecting page offset...")
    for i in range(min(30, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if not lines: continue
        if re.fullmatch(r"\d{1,3}", lines[-1]):
            offset = i - (int(lines[-1]) - 1)
            print(f"   ✅ Offset detected: {offset}")
            return offset
    return 0

# extract_toc
def extract_toc(pdf, offset, max_pages=15):
    """Parses TOC to find section bounds."""
    print(f"📚 Step 2: Reading TOC (Pages 1-{max_pages})...")
    toc = []
    for i in range(min(max_pages, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        for line in text.split("\n"):
            m = re.search(r"^(.*?)(\.+)\s*(\d+)$", line.strip())
            if m:
                toc.append({"title": m.group(1).strip().upper(), "pdf_page": int(m.group(3)) - 1 + offset})
    return toc

# find_section_page_range
def find_section_page_range(toc, aliases):
    """Identifies the start and end PDF pages for 'OUR BUSINESS'."""
    aliases = [a.upper() for a in aliases]
    matches = [t for t in toc if t["title"] in aliases]
    if not matches: return None
    start = min(t["pdf_page"] for t in matches)
    later = sorted(t["pdf_page"] for t in toc if t["pdf_page"] > start)
    end = later[0] - 1 if later else start + 30
    print(f"   📂 Section mapped to PDF Pages {start+1} to {end+1}")
    return start, end

# extract_paragraphs_robust
def extract_paragraphs_robust(pdf, start_page, end_page, start_markers, end_markers):
    """
    Extracts text between flexible start and end markers using case-insensitive matching.
    """
    print(f"📄 Step 3: Scanning for start markers {start_markers} to end markers {end_markers}...")
    all_paragraphs = []
    capture = False
    current_block = []
    para_count = 1

    # Compile regex patterns for efficiency and case-insensitivity
    start_patterns = [re.compile(re.escape(m), re.IGNORECASE) for m in start_markers]
    end_patterns = [re.compile(re.escape(m), re.IGNORECASE) for m in end_markers]

    for i in range(start_page, min(end_page + 1, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        lines = text.split('\n')
        current_page_num = i + 1

        for line in lines:
            clean = line.strip()
            if not clean: continue

            # Check if current line matches any Start Marker
            if not capture:
                if any(pattern.search(clean) for pattern in start_patterns) and len(clean) < 80:
                    capture = True
                    continue

            # Check if current line matches any End Marker
            if capture:
                if any(pattern.search(clean) for pattern in end_patterns) and len(clean) < 80:
                    capture = False
                    break

            if capture:
                # Paragraph logic: Detect start of new block (bullets or all-caps headers)
                if re.match(r'^[•▪\*]|^[A-Z][^a-z]+$', clean) and current_block:
                    all_paragraphs.append({
                        "paragraph_no": para_count,
                        "page_no": current_page_num,
                        "text": " ".join(current_block)
                    })
                    current_block = [clean]
                    para_count += 1
                else:
                    current_block.append(clean)

        if not capture and all_paragraphs: break

    if current_block:
        all_paragraphs.append({
            "paragraph_no": para_count,
            "page_no": current_page_num,
            "text": " ".join(current_block)
        })

    final_paras = [p for p in all_paragraphs if len(p["text"].split()) > 15]
    print(f"   ✅ Captured {len(final_paras)} distinct paragraph blocks.")
    return final_paras

# convert_to_verbatim_compliance
def convert_to_verbatim_compliance(text):
    """Deterministic conversion of 1P->3P and units."""
    text = text.replace("million", "crore").replace("billion", "crore")
    rules = [
        (r"\bOur Company\b", "The Company"), (r"\bour company\b", "the Company"),
        (r"\bWe are\b", "The Company is"), (r"\bwe are\b", "the Company is"),
        (r"\bWe\b", "The Company"), (r"\bwe\b", "the Company"),
        (r"\bOur\b", "The Company’s"), (r"\bour\b", "the Company’s")
    ]
    for pat, rep in rules:
        text = re.sub(pat, rep, text)
    return " ".join(text.split())

# gemini_select_paragraphs
def gemini_select_paragraphs(paragraphs, target_count=12):
    """Selects best paragraphs with error handling and dynamic count."""
    actual_target = min(target_count, len(paragraphs))
    print(f"🤖 Step 4: Gemini selecting top {actual_target} paragraph blocks...")

    lines = [f"PARA_{p['paragraph_no']}: {p['converted_text']}" for p in paragraphs]

    prompt = (
        f"Return a JSON list of IDs for the top {actual_target} most detailed strengths. "
        f"Format: [\"PARA_1\", \"PARA_2\"]. "
        f"Return ONLY the JSON list.\n\n"
        f"PARAGRAPHS:\n" + "\n".join(lines)
    )

    try:
        # response_mime_type: "application/json" is the most robust way to avoid parsing errors
        resp = MODEL.generate_content(
            prompt,
            generation_config={
                "temperature": 0.0,
                "response_mime_type": "application/json"
            }
        )

        # We don't need heavy regex if we use application/json, but we'll keep a simple one for safety
        clean_text = resp.text.strip().replace('```json', '').replace('```', '')
        return json.loads(clean_text)

    except Exception as e:
        print(f"⚠️ Gemini JSON parsing failed: {e}. Falling back to available paragraphs.")
        return [f"PARA_{p['paragraph_no']}" for p in paragraphs[:actual_target]]

# collate_verbatim_report
def collate_verbatim_report(selected_blocks, company_name):
    """Formatting final report with strict verbatim rules."""
    print("📝 Step 5: Formatting final report (Strict Verbatim Mode)...")
    formatted_input = "\n\n".join([f"[BLOCK {p['paragraph_no']}]: {p['converted_text']}" for p in selected_blocks])
    prompt = f"""
    You are a compliance officer for {company_name}.
    TASK: Organize the provided paragraphs into EXACTLY 8 points.

    STRICT RULES:
    1. Each point must have a **BOLD TITLE**.
    2. Under the title, place the relevant paragraphs as ONE continuous block of 4-6 lines.
    3. DO NOT break the internal flow of the paragraphs. Keep examples attached to their descriptions.
    4. NO SUB-BULLETS. NO NESTED LISTS.
    5. Convert 'million' to 'crore' and 'billion' to 'crore'.
    6. Use the text VERBATIM.
    7. DO NOT paraphrase, rephrase, shorten, expand, or reorder the content.
    8. DO NOT add, remove, or change any facts, numbers, dates, or entity names.
    9. Fix ONLY grammar, punctuation, and awkward repetition of references like "The Company", "it", "its", and "{company_name}".
    10. Make the use of "{company_name}", "the Company", and "it/its" sound natural and varied.

    TEXT: {formatted_input}
    """

    resp = MODEL.generate_content(prompt, generation_config={"temperature": 0.0})
    return resp.text.strip()

# entire pipeline block
# --------------------------------------------------------------------------

print(f"🚀 Running Pipeline for {COMPANY_NAME}...\n" + "="*50)

with pdfplumber.open(PDF_PATH) as pdf:
    offset = detect_page_offset(pdf)
    toc = extract_toc(pdf, offset)
    p_range = find_section_page_range(toc, CO_TOC_TITLES)

    # # Pass the lists instead of single strings
    # para_data = extract_paragraphs_robust(
    #     pdf,
    #     p_range[0],
    #     p_range[1],
    #     START_MARKERS,
    #     END_MARKERS
    # )

    # Try extraction with END_MARKERS
    para_data = extract_paragraphs_robust(
        pdf,
        p_range[0],
        p_range[1],
        START_MARKERS,
        END_MARKERS
    )

    # Condition 1: No END_MARKERS provided
    no_end_markers_defined = not END_MARKERS

    # Condition 2: END_MARKERS provided but not found in text
    end_markers_not_found = END_MARKERS and len(para_data) > 15

    if no_end_markers_defined or end_markers_not_found:
        print("⚠️ End markers not defined or not recognised. Applying 15-paragraph cap.")

        if len(para_data) > 15:
            para_data = para_data[:15]




if para_data:
    for p in para_data:
        p["converted_text"] = convert_to_verbatim_compliance(p["text"])
        p["id_key"] = f"PARA_{p['paragraph_no']}"

    selected_id_keys = gemini_select_paragraphs(para_data, target_count=12)
    selected_blocks = [p for p in para_data if p["id_key"] in selected_id_keys]
    final_output = collate_verbatim_report(selected_blocks, COMPANY_NAME)

    # 💾 SAVING ALL 4 FILES WITH TRACKING
    print("💾 Saving all 4 output files with Page/Paragraph tracking...")

    # File 1: strengths_conversion_audit.csv (Updated with Tracking Columns)
    pd.DataFrame(para_data)[[
        "paragraph_no", "page_no", "text", "converted_text"
    ]].to_csv(os.path.join(OUTPUT_DIR, "strengths_conversion_audit.csv"), index=False)

    # File 2: strengths_selection_audit.csv (Updated with Tracking Columns)
    sel_audit = []
    for p in para_data:
        sel_audit.append({
            "paragraph_no": p["paragraph_no"],
            "page_no": p["page_no"],
            "included_in_final": (1 if p["id_key"] in selected_id_keys else 0),
            "full_converted_text": p["converted_text"]
        })
    pd.DataFrame(sel_audit).to_csv(os.path.join(OUTPUT_DIR, "strengths_selection_audit.csv"), index=False)

    # File 3 & 4: Text files
    raw_summary = "\n\n".join([p["converted_text"] for p in selected_blocks])
    with open(os.path.join(OUTPUT_DIR, "strengths_summary_raw.txt"), "w") as f: f.write(raw_summary)
    with open(os.path.join(OUTPUT_DIR, "strengths_summary_polished.txt"), "w") as f: f.write(final_output)

    print("\n✅ Success. Audits now include 'paragraph_no' and 'page_no' tracking.")
    print("\n" + "="*50 + "\n📜 FINAL VERBATIM SUMMARY\n" + "="*50)
    print(final_output)
# >>>>>>  IMPORTANT  <<<<<<
# Paste full Strengths script here exactly as-is (except its own Cell 0).

strengths_final_text = final_output

print("✅ Module 3 Completed.")


🚀 STARTING MODULE 3: STRENGTHS
🚀 Running Pipeline for Meesho...
🔍 Step 1: Detecting page offset...
   ✅ Offset detected: 6
📚 Step 2: Reading TOC (Pages 1-15)...
   📂 Section mapped to PDF Pages 306 to 348
📄 Step 3: Scanning for start markers ['Our Strengths', 'Our Competitive Strengths'] to end markers ['Our Strategies', 'Our Strategy', 'Our Growth Strategies']...
   ✅ Captured 9 distinct paragraph blocks.
🤖 Step 4: Gemini selecting top 9 paragraph blocks...
📝 Step 5: Formatting final report (Strict Verbatim Mode)...
💾 Saving all 4 output files with Page/Paragraph tracking...

✅ Success. Audits now include 'paragraph_no' and 'page_no' tracking.

📜 FINAL VERBATIM SUMMARY
Here is the organized text, adhering to all strict rules:

**1. Meesho's Self-Reinforcing Flywheel Platform**
Meesho's platform is built on multiple scaled self-reinforcing flywheels, orchestrating transactions among consumers, sellers, logistics partners, and content creators. At its core is the commerce flywheel: wid

In [ ]:
# strengths_final_text
# kmp_final_text
# company_overview_final_text

In [ ]:
# ============================================================
# ============================================================
#               MODULE 4 — STRATEGIES
# ============================================================
# ============================================================

# print("\n" + "="*70)
print("🚀 STARTING MODULE 4: STRATEGIES")
# print("="*70)

SECTION_NAME = "Strategies"
CO_TOC_TITLES = ["OUR BUSINESS"]

START_MARKERS = ["Our Strategies","Our Strategy", "Our Growth Strategies","Our Growth Strategy"]
END_MARKERS = ["Our Corporate Structure", "CORPORATE STRUCTURE", "Our Technology"]

# (All helper functions copied EXACTLY from strategy_ver1.py)
# --------------------------------------------------------------------------
# detect_page_offset
def detect_page_offset(pdf):
    """Detects printed page number vs PDF index."""
    print("🔍 Step 1: Detecting page offset...")
    for i in range(min(30, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if not lines: continue
        if re.fullmatch(r"\d{1,3}", lines[-1]):
            offset = i - (int(lines[-1]) - 1)
            print(f"   ✅ Offset detected: {offset}")
            return offset
    return 0

# extract_toc
def extract_toc(pdf, offset, max_pages=15):
    """Parses TOC to find section bounds."""
    print(f"📚 Step 2: Reading TOC (Pages 1-{max_pages})...")
    toc = []
    for i in range(min(max_pages, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        for line in text.split("\n"):
            m = re.search(r"^(.*?)(\.+)\s*(\d+)$", line.strip())
            if m:
                toc.append({"title": m.group(1).strip().upper(), "pdf_page": int(m.group(3)) - 1 + offset})
    return toc

# find_section_page_range
def find_section_page_range(toc, aliases):
    """Identifies the start and end PDF pages for 'OUR BUSINESS'."""
    aliases = [a.upper() for a in aliases]
    matches = [t for t in toc if t["title"] in aliases]
    if not matches: return None
    start = min(t["pdf_page"] for t in matches)
    later = sorted(t["pdf_page"] for t in toc if t["pdf_page"] > start)
    end = later[0] - 1 if later else start + 30
    print(f"   📂 Section mapped to PDF Pages {start+1} to {end+1}")
    return start, end

# extract_paragraphs_robust
def extract_paragraphs_robust(pdf, start_page, end_page, start_markers, end_markers):
    """
    Extracts text between flexible start and end markers using case-insensitive matching.
    """
    print(f"📄 Step 3: Scanning for start markers {start_markers} to end markers {end_markers}...")
    all_paragraphs = []
    capture = False
    current_block = []
    para_count = 1

    # Compile regex patterns for efficiency and case-insensitivity
    start_patterns = [re.compile(re.escape(m), re.IGNORECASE) for m in start_markers]
    end_patterns = [re.compile(re.escape(m), re.IGNORECASE) for m in end_markers]

    for i in range(start_page, min(end_page + 1, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        lines = text.split('\n')
        current_page_num = i + 1

        for line in lines:
            clean = line.strip()
            if not clean: continue

            # Check if current line matches any Start Marker
            if not capture:
                if any(pattern.search(clean) for pattern in start_patterns) and len(clean) < 80:
                    capture = True
                    continue

            # Check if current line matches any End Marker
            if capture:
                if any(pattern.search(clean) for pattern in end_patterns) and len(clean) < 80:
                    capture = False
                    break

            if capture:
                # Paragraph logic: Detect start of new block (bullets or all-caps headers)
                if re.match(r'^[•▪\*]|^[A-Z][^a-z]+$', clean) and current_block:
                    all_paragraphs.append({
                        "paragraph_no": para_count,
                        "page_no": current_page_num,
                        "text": " ".join(current_block)
                    })
                    current_block = [clean]
                    para_count += 1
                else:
                    current_block.append(clean)

        if not capture and all_paragraphs: break

    if current_block:
        all_paragraphs.append({
            "paragraph_no": para_count,
            "page_no": current_page_num,
            "text": " ".join(current_block)
        })

    final_paras = [p for p in all_paragraphs if len(p["text"].split()) > 15]
    print(f"   ✅ Captured {len(final_paras)} distinct paragraph blocks.")
    return final_paras

# convert_to_verbatim_compliance
def convert_to_verbatim_compliance(text):
    """Deterministic conversion of 1P->3P and units."""
    text = text.replace("million", "crore").replace("billion", "crore")
    rules = [
        (r"\bOur Company\b", "The Company"), (r"\bour company\b", "the Company"),
        (r"\bWe are\b", "The Company is"), (r"\bwe are\b", "the Company is"),
        (r"\bWe\b", "The Company"), (r"\bwe\b", "the Company"),
        (r"\bOur\b", "The Company’s"), (r"\bour\b", "the Company’s")
    ]
    for pat, rep in rules:
        text = re.sub(pat, rep, text)
    return " ".join(text.split())

# gemini_select_paragraphs
def gemini_select_paragraphs(paragraphs, target_count=12):
    """Selects best paragraphs with error handling and dynamic count."""
    actual_target = min(target_count, len(paragraphs))
    print(f"🤖 Step 4: Gemini selecting top {actual_target} paragraph blocks...")

    lines = [f"PARA_{p['paragraph_no']}: {p['converted_text']}" for p in paragraphs]

    prompt = (
        f"Return a JSON list of IDs for the top {actual_target} most detailed strategies. "
        f"Format: [\"PARA_1\", \"PARA_2\"]. "
        f"Return ONLY the JSON list.\n\n"
        f"PARAGRAPHS:\n" + "\n".join(lines)
    )

    try:
        # response_mime_type: "application/json" is the most robust way to avoid parsing errors
        resp = MODEL.generate_content(
            prompt,
            generation_config={
                "temperature": 0.0,
                "response_mime_type": "application/json"
            }
        )

        # We don't need heavy regex if we use application/json, but we'll keep a simple one for safety
        clean_text = resp.text.strip().replace('```json', '').replace('```', '')
        return json.loads(clean_text)

    except Exception as e:
        print(f"⚠️ Gemini JSON parsing failed: {e}. Falling back to available paragraphs.")
        return [f"PARA_{p['paragraph_no']}" for p in paragraphs[:actual_target]]

# collate_verbatim_report
def collate_verbatim_report(selected_blocks, company_name):
    """Formatting final report with strict verbatim rules."""
    print("📝 Step 5: Formatting final report (Strict Verbatim Mode)...")
    formatted_input = "\n\n".join([f"[BLOCK {p['paragraph_no']}]: {p['converted_text']}" for p in selected_blocks])
    prompt = f"""
    You are a compliance officer for {company_name}.
    TASK: Organize the provided paragraphs into EXACTLY 8 points.

    STRICT RULES:
    1. Each point must have a **BOLD TITLE**.
    2. Under the title, place the relevant paragraphs as ONE continuous block of 4-6 lines.
    3. DO NOT break the internal flow of the paragraphs. Keep examples attached to their descriptions.
    4. NO SUB-BULLETS. NO NESTED LISTS.
    5. Convert 'million' to 'crore' and 'billion' to 'crore'.
    6. Use the text VERBATIM.
    7. DO NOT paraphrase, rephrase, shorten, expand, or reorder the content.
    8. DO NOT add, remove, or change any facts, numbers, dates, or entity names.
    9. Fix ONLY grammar, punctuation, and awkward repetition of references like "The Company", "it", "its", and "{company_name}".
    10. Make the use of "{company_name}", "the Company", and "it/its" sound natural and varied.

    TEXT: {formatted_input}
    """

    resp = MODEL.generate_content(prompt, generation_config={"temperature": 0.0})
    return resp.text.strip()

# entire pipeline block
# --------------------------------------------------------------------------

# >>>>>>  IMPORTANT  <<<<<<
# Paste full Strategies script here exactly as-is (except its own Cell 0).
print(f"🚀 Running Pipeline for {COMPANY_NAME}...\n" + "="*50)

with pdfplumber.open(PDF_PATH) as pdf:
    offset = detect_page_offset(pdf)
    toc = extract_toc(pdf, offset)
    p_range = find_section_page_range(toc, CO_TOC_TITLES)

    # # Pass the lists instead of single strings
    # para_data = extract_paragraphs_robust(
    #     pdf,
    #     p_range[0],
    #     p_range[1],
    #     START_MARKERS,
    #     END_MARKERS
    # )


    # Try extraction with END_MARKERS
    para_data = extract_paragraphs_robust(
        pdf,
        p_range[0],
        p_range[1],
        START_MARKERS,
        END_MARKERS
    )

    # Condition 1: No END_MARKERS provided
    no_end_markers_defined = not END_MARKERS

    # Condition 2: END_MARKERS provided but not found in text
    end_markers_not_found = END_MARKERS and len(para_data) > 25

    if no_end_markers_defined or end_markers_not_found:
        print("⚠️ End markers not defined or not recognised. Applying 25-paragraph cap.")

        if len(para_data) > 25:
            para_data = para_data[:25]




if para_data:
    for p in para_data:
        p["converted_text"] = convert_to_verbatim_compliance(p["text"])
        p["id_key"] = f"PARA_{p['paragraph_no']}"

    selected_id_keys = gemini_select_paragraphs(para_data, target_count=12)

    selected_blocks = [p for p in para_data if p["id_key"] in selected_id_keys]
    final_output = collate_verbatim_report(selected_blocks, COMPANY_NAME)

    # 💾 SAVING ALL 4 FILES WITH TRACKING
    print("💾 Saving all 4 output files with Page/Paragraph tracking...")

    # File 1: strengths_conversion_audit.csv (Updated with Tracking Columns)
    pd.DataFrame(para_data)[[
        "paragraph_no", "page_no", "text", "converted_text"
    ]].to_csv(os.path.join(OUTPUT_DIR, "strategies_conversion_audit.csv"), index=False)

    # File 2: strengths_selection_audit.csv (Updated with Tracking Columns)
    sel_audit = []
    for p in para_data:
        sel_audit.append({
            "paragraph_no": p["paragraph_no"],
            "page_no": p["page_no"],
            "included_in_final": (1 if p["id_key"] in selected_id_keys else 0),
            "full_converted_text": p["converted_text"]
        })
    pd.DataFrame(sel_audit).to_csv(os.path.join(OUTPUT_DIR, "strategies_selection_audit.csv"), index=False)

    # File 3 & 4: Text files
    raw_summary = "\n\n".join([p["converted_text"] for p in selected_blocks])
    with open(os.path.join(OUTPUT_DIR, "strategies_summary_raw.txt"), "w") as f: f.write(raw_summary)
    with open(os.path.join(OUTPUT_DIR, "strategies_summary_polished.txt"), "w") as f: f.write(final_output)

    print("\n✅ Success. Audits now include 'paragraph_no' and 'page_no' tracking.")
    print("\n" + "="*50 + "\n📜 FINAL VERBATIM SUMMARY\n" + "="*50)
    print(final_output)
strategies_final_text = final_output

print("✅ Module 4 Completed.")

🚀 STARTING MODULE 4: STRATEGIES
🚀 Running Pipeline for Meesho...
🔍 Step 1: Detecting page offset...
   ✅ Offset detected: 6
📚 Step 2: Reading TOC (Pages 1-15)...
   📂 Section mapped to PDF Pages 306 to 348
📄 Step 3: Scanning for start markers ['Our Strategies', 'Our Strategy', 'Our Growth Strategies', 'Our Growth Strategy'] to end markers ['Our Corporate Structure', 'CORPORATE STRUCTURE', 'Our Technology']...
   ✅ Captured 4 distinct paragraph blocks.
🤖 Step 4: Gemini selecting top 4 paragraph blocks...
📝 Step 5: Formatting final report (Strict Verbatim Mode)...
💾 Saving all 4 output files with Page/Paragraph tracking...

✅ Success. Audits now include 'paragraph_no' and 'page_no' tracking.

📜 FINAL VERBATIM SUMMARY
Here is the organized text, adhering to all strict rules:

**Expanding Consumer Base and Market Penetration**
Increase consumer base and their transaction frequency by expanding Meesho’s product listings and seller base. Meesho’s aim is to democratise internet commerce for e

In [ ]:
# ============================================================
# ============================================================
#          FINAL STEP — GENERATE IPO REPORT (.DOCX)
# ============================================================
# ============================================================

print("\n" + "="*70)
print("📝 GENERATING FINAL IPO REPORT DOCUMENT")
print("="*70)

document = Document()

# Title
title = document.add_heading(f"{COMPANY_NAME} IPO Report", level=0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Section 1
document.add_heading("Company Overview", level=1)
document.add_paragraph(company_overview_final_text)

# Section 2
document.add_heading("Key Management Personnel", level=1)
document.add_paragraph(kmp_final_text)

# Section 3
document.add_heading("Strengths", level=1)
document.add_paragraph(strengths_final_text)

# Section 4
document.add_heading("Strategies", level=1)
document.add_paragraph(strategies_final_text)

# Save File
final_doc_path = os.path.join(OUTPUT_DIR, f"{COMPANY_NAME}_IPO_Report.docx")
document.save(final_doc_path)

print(f"📄 Final IPO Report saved at:\n{final_doc_path}")
print("\n🎉 COMPLETE PIPELINE EXECUTED SUCCESSFULLY.")


📝 GENERATING FINAL IPO REPORT DOCUMENT
📄 Final IPO Report saved at:
/content/ipo_outputs/Meesho_IPO_Report.docx

🎉 COMPLETE PIPELINE EXECUTED SUCCESSFULLY.


In [ ]:
from google.colab import files
files.download(final_doc_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import glob

print("\n📂 Downloading all audit files...\n")

for file in glob.glob(os.path.join(OUTPUT_DIR, "*.csv")):
    print("⬇ Downloading:", file)
    files.download(file)


📂 Downloading all audit files...

⬇ Downloading: /content/ipo_outputs/strengths_selection_audit.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading: /content/ipo_outputs/strategies_selection_audit.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading: /content/ipo_outputs/kmp_profiles_audit.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>